# Phase 2 — Homology Search & Candidate Discovery

This notebook takes the database built in **Phase 1** (`search_db.fasta`) and
searches it for homologs of a query sequence using three independent,
complementary methods:

| Method | Nature | Why included |
| --- | --- | --- |
| **PSI-BLAST** | Iterative, position-specific scoring | Sensitive to divergent homologs via a query-derived profile |
| **jackhmmer** | Iterative profile-HMM search (HMMER) | Different sensitivity profile; catches homologs PSI-BLAST may miss |
| **hmmsearch** | Single pass vs. curated Pfam `PF02346` | Independent, expert-curated check not derived from the query |

Using three methods guards against method-specific blind spots. Hits are
unioned into one non-redundant candidate set, with a provenance table
recording exactly which method(s) found each sequence. Every search step is
checkpointed on its output file.

**Steps**

0. Setup & configuration
1. Prerequisites — query FASTA & required files
2. Build the BLAST database
3. Run PSI-BLAST
4. Run jackhmmer
5. Run hmmsearch (PF02346)
6. Merge hits & write candidate outputs

## Step 0 — Setup & configuration

All tunable parameters live in one frozen `Config`: E-value thresholds,
iteration counts, scoring matrix, file paths, and the `wsl` command prefix
used to call BLAST+/HMMER. Centralising the `wsl` prefix means the whole
notebook can be pointed at a native Linux environment by changing one field.

**Paths and WSL.** Every `Config` path is a `pathlib.Path`, which renders
with backslashes on Windows. Because these tools run *inside* WSL, backslashes
are literal characters there, not separators — so any path passed to a
`wsl`-prefixed command uses `.as_posix()`. The one exception is `log_file=`
in `run_command`, opened by Python on the Windows side, which stays a plain
`Path`.

Shared parsers come from `seqio_utils` (`parse_blast_tabular`,
`parse_hmmer_tabular`, `parse_fasta_by_accession`), so Phases 2 and 3 parse
search output identically.

`run_command` wraps `subprocess.run`: it logs the exact command, captures
stdout/stderr to a log file, and raises a clear error on non-zero exit.

In [1]:
from __future__ import annotations

import csv
import logging
import subprocess
from dataclasses import dataclass, field
from pathlib import Path

from seqio_utils import parse_blast_tabular, parse_hmmer_tabular, parse_fasta_by_accession

In [2]:
@dataclass(frozen=True)
class Config:
    wsl_prefix: list[str] = field(default_factory=lambda: ["wsl"])

    search_db_fasta: Path = Path("output_phase1/search_db.fasta")
    pfam_hmm: Path = Path("PF02346.hmm")

    query_acc: str = "Q77HM6_A29L_MPXV"
    query_seq: str = (
        "MDGTLFPGDDDLAIPATEFFSTKAAKNPETKREAIVKAYGDDNEETLKQRLTNLEKKITNITTKFEQIEK"
        "CCKHNDEVLFRLENHAETLRAAMISLAKKIDVQTGRRPYE"
    )

    num_iterations: int = 5
    evalue: str = "1e-3"
    inclusion_ethresh: str = "1e-3"
    matrix: str = "BLOSUM62"
    gapopen: int = 11
    gapextend: int = 1

    out_dir: Path = field(default_factory=lambda: Path("output_phase2"))

    @property
    def query_fasta(self) -> Path:
        return self.out_dir / "query.fasta"

    @property
    def blastdb_name(self) -> Path:
        return self.out_dir / "searchdb"

    @property
    def psiblast_tbl(self) -> Path:
        return self.out_dir / "psiblast.tbl"

    @property
    def jackhmmer_tbl(self) -> Path:
        return self.out_dir / "jackhmmer_tbl.txt"

    @property
    def hmmsearch_tbl(self) -> Path:
        return self.out_dir / "hmmsearch_tbl.txt"

    @property
    def candidates_fasta(self) -> Path:
        return self.out_dir / "homology_candidates.fasta"

    @property
    def provenance_csv(self) -> Path:
        return self.out_dir / "search_hits_provenance.csv"


CFG = Config()
CFG.out_dir.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(CFG.out_dir / "phase2.log"),
    ],
)
log = logging.getLogger("phase2")

In [3]:
class CommandError(RuntimeError):
    """Raised when an external tool exits with a non-zero status."""


def run_command(cmd: list[str], log_file: Path) -> None:
    log.info("Running: %s", " ".join(cmd))
    with open(log_file, "w", encoding="utf-8") as fh:
        result = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT)
    if result.returncode != 0:
        raise CommandError(
            f"Command failed (exit {result.returncode}): {' '.join(cmd)}. "
            f"See {log_file} for details."
        )

## Step 1 — Prerequisites: query FASTA & required files

Writes the query to `query.fasta`, then verifies the two required inputs
exist before any tool launches: `search_db.fasta` (from Phase 1) and
`PF02346.hmm` (downloaded from Pfam/InterPro). Failing fast here with a clear
message is far easier to diagnose than a cryptic tool error several steps
later.

In [4]:
with open(CFG.query_fasta, "w", encoding="utf-8") as fh:
    fh.write(f">{CFG.query_acc}\n{CFG.query_seq}\n")
log.info("Wrote query FASTA: %s", CFG.query_fasta)

missing = [p for p in (CFG.search_db_fasta, CFG.pfam_hmm) if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing required input file(s): {[str(p) for p in missing]}. "
        "search_db.fasta comes from Phase 1; PF02346.hmm must be downloaded "
        "from Pfam/InterPro before continuing."
    )
log.info("All prerequisite files found.")

2026-07-22 11:23:37,670 [INFO] Wrote query FASTA: output_phase2\query.fasta
2026-07-22 11:23:37,673 [INFO] All prerequisite files found.


## Step 2 — Build the BLAST database

Builds a protein BLAST database from `search_db.fasta` with `makeblastdb`.
Checkpointed on the `.pin` index file — if the database was already built,
this step is skipped.

In [5]:
blastdb_marker = CFG.blastdb_name.with_suffix(".pin")

if blastdb_marker.exists():
    log.info("BLAST database already built (%s found) — skipping.", blastdb_marker)
else:
    log.info("Building BLAST database ...")
    run_command(
        CFG.wsl_prefix + [
            "makeblastdb", "-dbtype", "prot",
            "-in", CFG.search_db_fasta.as_posix(), "-out", CFG.blastdb_name.as_posix(),
        ],
        log_file=CFG.out_dir / "makeblastdb.log",
    )
    log.info("DONE — BLAST database: %s", CFG.blastdb_name)

2026-07-22 11:23:37,725 [INFO] Building BLAST database ...
2026-07-22 11:23:37,727 [INFO] Running: wsl makeblastdb -dbtype prot -in output_phase1/search_db.fasta -out output_phase2/searchdb
2026-07-22 11:23:39,622 [INFO] DONE — BLAST database: output_phase2\searchdb


## Step 3 — Run PSI-BLAST

Runs iterative PSI-BLAST (5 iterations) against the Step 2 database and parses
the tabular output (`-outfmt 6`) into the set of unique subject accessions.
The shared `parse_blast_tabular` tolerates PSI-BLAST's per-iteration blank
lines, `#`-comment lines, and the `Search has CONVERGED!` marker, so the
union across iterations is collected cleanly. Checkpointed on `psiblast.tbl`.

In [6]:
if CFG.psiblast_tbl.exists():
    log.info("%s exists — skipping PSI-BLAST.", CFG.psiblast_tbl)
else:
    log.info("Running PSI-BLAST ...")
    run_command(
        CFG.wsl_prefix + [
            "psiblast", "-query", CFG.query_fasta.as_posix(), "-db", CFG.blastdb_name.as_posix(),
            "-num_iterations", str(CFG.num_iterations),
            "-evalue", CFG.evalue, "-inclusion_ethresh", CFG.inclusion_ethresh,
            "-matrix", CFG.matrix,
            "-gapopen", str(CFG.gapopen), "-gapextend", str(CFG.gapextend),
            "-outfmt", "6", "-out", CFG.psiblast_tbl.as_posix(),
        ],
        log_file=CFG.out_dir / "psiblast.log",
    )

psiblast_hits = parse_blast_tabular(CFG.psiblast_tbl)
log.info("PSI-BLAST found %d hits.", len(psiblast_hits))

2026-07-22 11:23:39,733 [INFO] Running PSI-BLAST ...
2026-07-22 11:23:39,736 [INFO] Running: wsl psiblast -query output_phase2/query.fasta -db output_phase2/searchdb -num_iterations 5 -evalue 1e-3 -inclusion_ethresh 1e-3 -matrix BLOSUM62 -gapopen 11 -gapextend 1 -outfmt 6 -out output_phase2/psiblast.tbl
2026-07-22 11:23:40,477 [INFO] PSI-BLAST found 82 hits.


## Step 4 — Run jackhmmer

Runs iterative profile-HMM search (`jackhmmer`, 5 iterations) with the query
against the same database, then parses the HMMER `--tblout` output with the
shared `parse_hmmer_tabular`, which skips the `#`-prefixed comment lines HMMER
writes at the top and bottom. Checkpointed on `jackhmmer_tbl.txt`.

In [7]:
if CFG.jackhmmer_tbl.exists():
    log.info("%s exists — skipping jackhmmer.", CFG.jackhmmer_tbl)
else:
    log.info("Running jackhmmer ...")
    run_command(
        CFG.wsl_prefix + [
            "jackhmmer", "-N", str(CFG.num_iterations),
            "-E", CFG.evalue, "--incE", CFG.inclusion_ethresh,
            "--tblout", CFG.jackhmmer_tbl.as_posix(), "-o", (CFG.out_dir / "jackhmmer.log").as_posix(),
            CFG.query_fasta.as_posix(), CFG.search_db_fasta.as_posix(),
        ],
        log_file=CFG.out_dir / "jackhmmer_run.log",
    )

jackhmmer_hits = parse_hmmer_tabular(CFG.jackhmmer_tbl)
log.info("jackhmmer found %d hits.", len(jackhmmer_hits))

2026-07-22 11:23:40,552 [INFO] Running jackhmmer ...
2026-07-22 11:23:40,553 [INFO] Running: wsl jackhmmer -N 5 -E 1e-3 --incE 1e-3 --tblout output_phase2/jackhmmer_tbl.txt -o output_phase2/jackhmmer.log output_phase2/query.fasta output_phase1/search_db.fasta
2026-07-22 11:23:42,470 [INFO] jackhmmer found 82 hits.


## Step 5 — Run hmmsearch (PF02346)

Runs a single-pass `hmmsearch` of the curated Pfam profile `PF02346` against
the database — an independent check that depends only on the profile's learned
pattern, not on the query sequence. Reuses `parse_hmmer_tabular`, since both
HMMER tools share the `--tblout` format. Checkpointed on `hmmsearch_tbl.txt`.

In [8]:
if CFG.hmmsearch_tbl.exists():
    log.info("%s exists — skipping hmmsearch.", CFG.hmmsearch_tbl)
else:
    log.info("Running hmmsearch with PF02346 ...")
    run_command(
        CFG.wsl_prefix + [
            "hmmsearch", "-E", CFG.evalue,
            "--tblout", CFG.hmmsearch_tbl.as_posix(), "-o", (CFG.out_dir / "hmmsearch.log").as_posix(),
            CFG.pfam_hmm.as_posix(), CFG.search_db_fasta.as_posix(),
        ],
        log_file=CFG.out_dir / "hmmsearch_run.log",
    )

hmmsearch_hits = parse_hmmer_tabular(CFG.hmmsearch_tbl)
log.info("hmmsearch found %d hits.", len(hmmsearch_hits))

2026-07-22 11:23:42,525 [INFO] Running hmmsearch with PF02346 ...
2026-07-22 11:23:42,527 [INFO] Running: wsl hmmsearch -E 1e-3 --tblout output_phase2/hmmsearch_tbl.txt -o output_phase2/hmmsearch.log PF02346.hmm output_phase1/search_db.fasta
2026-07-22 11:23:43,185 [INFO] hmmsearch found 82 hits.


## Step 6 — Merge hits & write candidate outputs

Unions the hit sets from all three methods into one non-redundant candidate
list, then writes **`homology_candidates.fasta`** (sequences for every
candidate found in the database) and **`search_hits_provenance.csv`** (per
candidate: length and a Y/blank flag per method, so single-method hits are
distinguishable from those corroborated by all three). Sequences are looked up
via the shared `parse_fasta_by_accession`, which returns an accession-keyed
dict for O(1) lookup — the right shape for this join, distinct from the
`(header, sequence)` list Phase 1 uses for hash-based deduplication.

In [9]:
union = psiblast_hits | jackhmmer_hits | hmmsearch_hits
log.info("Total unique candidates across all 3 methods: %d", len(union))

db = parse_fasta_by_accession(CFG.search_db_fasta)

missing_from_db = sorted(acc for acc in union if acc not in db)
if missing_from_db:
    log.warning(
        "%d candidate accession(s) not found in %s — check accession formatting: %s",
        len(missing_from_db), CFG.search_db_fasta, missing_from_db[:5],
    )

with open(CFG.candidates_fasta, "w", encoding="utf-8") as fasta_out, \
     open(CFG.provenance_csv, "w", newline="", encoding="utf-8") as csv_out:
    writer = csv.writer(csv_out)
    writer.writerow(["acc", "length", "psiblast", "jackhmmer", "hmmsearch_PF02346"])

    for acc in sorted(union):
        seq = db.get(acc, "")
        if seq:
            fasta_out.write(f">{acc}\n{seq}\n")
        writer.writerow([
            acc, len(seq),
            "Y" if acc in psiblast_hits else "",
            "Y" if acc in jackhmmer_hits else "",
            "Y" if acc in hmmsearch_hits else "",
        ])

log.info("DONE — %s, %s", CFG.candidates_fasta, CFG.provenance_csv)

2026-07-22 11:23:43,232 [INFO] Total unique candidates across all 3 methods: 82
2026-07-22 11:23:43,271 [INFO] DONE — output_phase2\homology_candidates.fasta, output_phase2\search_hits_provenance.csv
